# Data Preprocessing & Feature Engineering

**Project**: Water Quality Prediction via Gradient Boosting & Deep Neural Networks  
**Stage**: 02. Preprocessing & Feature Selection  
**Deliverables**: Cleaned dataset, engineered environmental features, leakage-free train/val/test splits, and scaled inputs for MLP training.

---

### Pipeline Overview
1. **Cleaning**: Neutralize sensor hardware sentinels (-999999), convert US Fahrenheit to Celsius, clip analytical noise below detection limit (< 0 mg/L -> 0.0), and drop ingestion duplicate rows.
2. **Feature Engineering**:
   - $N:P$ nutrient stoichiometry ratio (`np_ratio`)
   - Organic pollution load ratio (`bod_do_ratio`)
   - Dissolved oxygen saturation deficit (`do_deficit`)
   - Cyclical seasonal encodings (`month_sin`, `month_cos`)
   - Ordinal target code for classification (`ccme_wqi_code`: 0 to 4)
3. **Stratified Splitting**: 70% Train, 15% Validation, 15% Test stratified by target class.
4. **Leakage-Free Imputation**: Group medians by `(country, waterbody_type)` fitted strictly on the training set.
5. **Feature Scaling**: Standard scaling fitted on training set for MLP neural network.


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dynamic project root discovery
for candidate in [".", "../..", ".."]:
    if os.path.exists(os.path.join(candidate, "data", "Dataset")):
        PROJECT_ROOT = os.path.abspath(candidate)
        break

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import (
    run_cleaning_pipeline,
    engineer_features,
    split_data,
    impute_missing_values,
    scale_features,
    prepare_training_datasets,
    BASE_NUMERIC_FEATURES,
    CCME_WQI_CATEGORIES,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL


## 1. Load & Clean Dataset
Load pre-cleaned dataset or execute the standardized cleaning pipeline.


In [2]:
cleaned_path = os.path.join(PROJECT_ROOT, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection", "cleaned_water_quality.parquet")

if os.path.exists(cleaned_path):
    print(f"Loading pre-cleaned dataset from {cleaned_path}...")
    df_clean = pd.read_parquet(cleaned_path)
else:
    print("Running cleaning pipeline from raw source...")
    raw_path = os.path.join(PROJECT_ROOT, "data", "Dataset", "Parquet", "Combined Data", "Combined_dataset.parquet")
    df_clean, summary = run_cleaning_pipeline(raw_path=raw_path, output_path=cleaned_path)

print(f"Cleaned dataset shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
df_clean.head(5)


Loading pre-cleaned dataset from /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/data/GeneratedOutput/02.Preprocessing&FeatureSelection/cleaned_water_quality.parquet...


Cleaned dataset shape: 4,287,890 rows x 23 columns


,country,area,waterbody_type,date,ammonia_mg_l,bod_mg_l,do_mg_l,orthophosphate_mg_l,ph,temperature_c,...,ccme_wqi,ph_flag_suspicious,temperature_c_flag_suspicious,ammonia_mg_l_flag_suspicious,bod_mg_l_flag_suspicious,do_mg_l_flag_suspicious,orthophosphate_mg_l_flag_suspicious,nitrogen_mg_l_flag_suspicious,nitrate_mg_l_flag_suspicious,ccme_values_flag_suspicious
0,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2014-12-01,0.083,NaN,4.15,0.062,NaN,24.7,...,Good,False,False,False,False,False,False,False,False,False
1,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-02-02,0.010,NaN,12.07,0.006,NaN,22.4,...,Excellent,False,False,False,False,False,False,False,False,False
2,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-03-02,0.013,NaN,10.59,0.026,NaN,17.0,...,Excellent,False,False,False,False,False,False,False,False,False
3,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-04-13,0.056,NaN,3.42,0.026,7.4,17.0,...,Excellent,False,False,False,False,False,False,False,False,False
4,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-01-15,0.086,NaN,3.91,0.075,7.4,23.2,...,Good,False,False,False,False,False,False,False,False,False


## 2. Feature Engineering
We compute stoichiometric ratios, oxygen saturation deficits, and seasonal cycles.


In [3]:
df_feat, new_cols = engineer_features(df_clean)
print(f"Engineered features added: {new_cols}")
df_feat[["country", "waterbody_type", "np_ratio", "bod_do_ratio", "do_deficit", "month_sin", "month_cos", "ccme_wqi_code"]].head()


2026-10-23 01:25:19,877 [INFO] Engineered 7 new features: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


Engineered features added: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


,country,waterbody_type,np_ratio,bod_do_ratio,do_deficit,month_sin,month_cos,ccme_wqi_code
0,Australia,River,13.043478,NaN,4.072801,-2.449294e-16,1.000000e+00,3
1,Australia,River,198.360656,NaN,0.000000,8.660254e-01,5.000000e-01,4
2,Australia,River,36.015326,NaN,0.000000,1.000000e+00,6.123234e-17,4
3,Australia,River,21.455939,NaN,6.185555,8.660254e-01,-5.000000e-01,4
4,Australia,River,10.119840,NaN,4.554795,5.000000e-01,8.660254e-01,3


## 3. Stratified Train / Validation / Test Splitting
Split data into 70% Train, 15% Validation, and 15% Test sets stratified by `ccme_wqi`.


In [4]:
train_raw, val_raw, test_raw = split_data(
    df_feat,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42
)

print(f"Train split: {len(train_raw):,} rows ({len(train_raw)/len(df_feat)*100:.1f}%)")
print(f"Val split:   {len(val_raw):,} rows ({len(val_raw)/len(df_feat)*100:.1f}%)")
print(f"Test split:  {len(test_raw):,} rows ({len(test_raw)/len(df_feat)*100:.1f}%)")


2026-10-23 01:25:37,549 [INFO] Split data: Train=3,001,523 (70.0%), Val=643,183 (15.0%), Test=643,184 (15.0%)


Train split: 3,001,523 rows (70.0%)
Val split:   643,183 rows (15.0%)
Test split:  643,184 rows (15.0%)


## 4. Leakage-Free Imputation & Feature Scaling
To prevent data leakage, imputation medians and scaling parameters are calculated solely on the training split.


In [5]:
features = [
    col for col in (BASE_NUMERIC_FEATURES + new_cols)
    if col in df_feat.columns and pd.api.types.is_numeric_dtype(df_feat[col])
]

# Impute missing values using train-set group medians
train_imp, val_imp, test_imp, imputer_info = impute_missing_values(
    train_raw, val_raw, test_raw, feature_cols=features
)

# Standard scale features for MLP neural net
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_imp, val_imp, test_imp, feature_cols=features
)

print(f"Number of modeling features: {len(features)}")
print(f"Train missing values remaining: {train_imp[features].isna().sum().sum()}")
print(f"Val missing values remaining:   {val_imp[features].isna().sum().sum()}")
print(f"Test missing values remaining:  {test_imp[features].isna().sum().sum()}")


Number of modeling features: 15
Train missing values remaining: 0
Val missing values remaining:   0
Test missing values remaining:  0


## 5. Artifact Verification & Inspection
Verify the saved datasets and metadata ready for Stage 03 Model Training.


In [6]:
out_dir = os.path.join(PROJECT_ROOT, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection")
meta_path = os.path.join(out_dir, "preprocessing_metadata.json")

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    tot = meta['total_rows']
    n_feats = len(meta['feature_columns'])
    feats = meta['feature_columns']
    classes = meta['target_classes']
    n_train = meta['split_counts']['train']
    n_val = meta['split_counts']['val']
    n_test = meta['split_counts']['test']
    print("Metadata summary:")
    print(f"  Total records: {tot:,}")
    print(f"  Features ({n_feats}): {feats}")
    print(f"  Target classes: {classes}")
    print(f"  Train split: {n_train:,}")
    print(f"  Val split:   {n_val:,}")
    print(f"  Test split:  {n_test:,}")

print("\nReady for model training (Tree baselines & Deep Neural Network).")


Metadata summary:
  Total records: 4,287,890
  Features (15): ['ph', 'temperature_c', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'nitrogen_mg_l', 'nitrate_mg_l', 'np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']
  Target classes: ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']
  Train split: 3,001,523
  Val split:   643,183
  Test split:  643,184

Ready for model training (Tree baselines & Deep Neural Network).
